# 02 - Production Random Forest Model Training
**Sepsis Clinical Decision Support System (CDSS)**  
*Script Source: `train_model.py` & `optimal_15_features/train_model_15.py`*

---

### Objectives:
1. **Feature Contract Enforcement**: Extract the 15 Optimal Clinical Features (`FEATURE_NAMES`) required by `services/ml_service.py`.
2. **Fallback Imputation Medians**: Calculate and serialize medians to `model/feature_medians.json` for live fallback inference.
3. **Stratified Train/Test Split**: 80% train, 20% test stratified on `SepsisLabel`.
4. **Random Forest Training**: Fit a 100-tree ensemble with `class_weight='balanced'` and `max_depth=15`.
5. **Evaluation**: Generate classification report, confusion matrix, and feature importances.
6. **Artifact Serialization & Startup Verification**: Save `sepsis_model.pkl` and test startup contract verification.

In [ ]:
# Ensure working directory is the project root regardless of how the notebook is opened
import os
import sys

if os.path.basename(os.getcwd()) in ['jupyter_notebooks', 'notebooks']:
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Active Project Root: {os.getcwd()}")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
import joblib

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries imported successfully!")

## Step 1: Feature Contract Definition
The CDSS backend specifies **15 Optimal Clinical Features**:
- **Vitals**: `HR`, `O2Sat`, `Temp`, `SBP`, `MAP`, `DBP`, `Resp`
- **Labs**: `WBC`, `Glucose`, `Lactate`, `Creatinine`, `Platelets`
- **Demographics & ICU Stay**: `Age`, `Gender`, `ICULOS`

In [ ]:
FEATURE_NAMES = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp",
    "WBC", "Glucose", "Lactate", "Creatinine", "Platelets",
    "Age", "Gender", "ICULOS"
]

print(f"Selected {len(FEATURE_NAMES)} optimal clinical features:")
for i, f in enumerate(FEATURE_NAMES, 1):
    print(f"  {i:2d}. {f}")

## Step 2: Loading Dataset & Exporting Feature Medians
We load `physionet_sepsis_processed.csv` (or fallback to `physionet_sepsis_combined.csv`). We calculate and save the population medians for the 15 features to `model/feature_medians.json`.

In [ ]:
csv_path = 'physionet_sepsis_processed.csv' if os.path.exists('physionet_sepsis_processed.csv') else 'physionet_sepsis_combined.csv'
print(f"Loading dataset from: {csv_path} ...")

df = pd.read_csv(csv_path)
print(f"Loaded {len(df):,} rows.")

# Extract 15 features and target
X = df[FEATURE_NAMES].copy()
y = df['SepsisLabel'].values

# Calculate and save feature medians
medians = X.median().to_dict()
os.makedirs('model', exist_ok=True)
with open('model/feature_medians.json', 'w') as f:
    json.dump(medians, f, indent=4)
print("Saved feature medians to 'model/feature_medians.json':")
for k, v in medians.items():
    print(f"  {k:12s}: {v:.2f}")

# Impute any remaining NaNs with column medians
X = X.fillna(medians)

## Step 3: Stratified Train / Test Split
Partition into 80% training and 20% testing sets stratified on `y` (`SepsisLabel`).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Samples: {len(X_train):,} (Sepsis cases: {y_train.sum():,}, {y_train.mean()*100:.2f}%)")
print(f"Testing Samples:  {len(X_test):,} (Sepsis cases: {y_test.sum():,}, {y_test.mean()*100:.2f}%)")

## Step 4: Training the Random Forest Classifier
We initialize the ensemble with:
- `n_estimators=100`: 100 decorrelated decision trees
- `max_depth=15`: Controls model complexity and prevents overfitting
- `class_weight='balanced'`: Automatically adjusts tree split weights inversely proportional to class frequencies
- `n_jobs=-1`: Parallelized training across all CPU cores

In [ ]:
print("Training Random Forest Classifier on 15 features...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print("Model training complete!")

## Step 5: Clinical Evaluation & Metrics
Evaluating model performance on the held-out test partition.

In [ ]:
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print("--- CLASSIFICATION REPORT ---")
print(classification_report(y_test, y_pred, digits=4))

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"PR-AUC Score:  {pr_auc:.4f}")

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False,
            xticklabels=['Pred Non-Sepsis (0)', 'Pred Sepsis (1)'],
            yticklabels=['True Non-Sepsis (0)', 'True Sepsis (1)'])
plt.title('Random Forest Confusion Matrix', fontsize=12, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Step 6: Feature Importance Analysis
Examine Mean Decrease in Impurity (Gini Importance) across the 15 clinical biomarkers.

In [ ]:
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_NAMES).sort_values()

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='#2b5c8f', edgecolor='navy')
plt.title('Random Forest Feature Importance (15 Biomarkers)', fontsize=13, fontweight='bold')
plt.xlabel('Gini Importance (MDI)')
plt.tight_layout()
plt.show()

print("Top 5 Clinical Drivers of Sepsis Risk:")
for feat, imp in feat_imp.tail(5).iloc[::-1].items():
    print(f"  - {feat:12s}: {imp:.4f}")

## Step 7: Model Serialization & CDSS Contract Check
We serialize the trained model artifact to `sepsis_model.pkl` and run startup contract validation against `services/ml_service.py`.

In [ ]:
model_save_path = 'sepsis_model.pkl'
joblib.dump(rf_model, model_save_path)
print(f"Saved trained production model to '{model_save_path}' ({os.path.getsize(model_save_path)/(1024*1024):.1f} MB).")

# Startup Contract Verification
loaded_model = joblib.load(model_save_path)
loaded_features = list(getattr(loaded_model, "feature_names_in_", []))

if loaded_features == FEATURE_NAMES:
    print("STARTUP CONTRACT VERIFIED: Saved model features match FEATURE_NAMES exactly!")
else:
    print(f"CONTRACT MISMATCH: Expected {FEATURE_NAMES}, got {loaded_features}")

# Test sample prediction with population medians
sample_df = pd.DataFrame([medians])
prob = loaded_model.predict_proba(sample_df)[0, 1]
print(f"Test inference with median vitals -> Predicted Sepsis Probability: {prob:.4f}")
print("Ready for live deployment in Flask Sepsis CDSS!")